In [276]:
import pandas as pd

true_df = pd.read_csv("../data/True.csv")
fake_df = pd.read_csv("../data/Fake.csv")

true_df = true_df.dropna(subset=["title", "text"])
fake_df = fake_df.dropna(subset=["title", "text"])

true_df["label"] = 1
fake_df["label"] = 0

df = pd.concat([true_df, fake_df], ignore_index=True)

df.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


In [277]:
df["label"].value_counts()

label
0    23481
1    21417
Name: count, dtype: int64

In [278]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [279]:
df["content"] = df["title"] + " " + df["text"]
df = df[["content", "label"]]

In [280]:
df

,content,label
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,0
1,Failed GOP Candidates Remembered In Hilarious...,0
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,0
3,California AG pledges to defend birth control ...,1
4,AZ RANCHERS Living On US-Mexico Border Destroy...,0
...,...,...
44893,Nigeria says U.S. agrees delayed $593 million ...,1
44894,Boiler Room #62 – Fatal Illusions Tune in to t...,0
44895,ATHEISTS SUE GOVERNOR OF TEXAS Over Display on...,0
44896,Republican tax plan would deal financial hit t...,1


In [281]:
X = df["content"]
y = df["label"]

In [ ]:
import re
import nltk

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
def preprocess_text(text):
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    
    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)
    
    # Remove mentions
    text = re.sub(r"@\w+", "", text)
    
    # Remove hashtags symbol but keep the word
    text = re.sub(r"#", "", text)
    
    # Remove punctuation and special characters
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    
    # Tokenization
    tokens = text.split()
    
    return " ".join(tokens)

In [285]:
X = X.apply(preprocess_text)

In [286]:
df_processed = pd.DataFrame({
    "content": X,
    "label": y
})

df_processed = df_processed.drop_duplicates(
    subset=["content"]
).reset_index(drop=True)

In [287]:
X = df_processed["content"]
y = df_processed["label"]

In [288]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

In [289]:
print(
    X_train.duplicated().sum(),
    X_val.duplicated().sum(),
    X_test.duplicated().sum()
)

0 0 0


In [290]:
train_texts = set(X_train)
val_texts = set(X_val)
test_texts = set(X_test)

print("Train-Val overlap:", len(train_texts & val_texts))
print("Train-Test overlap:", len(train_texts & test_texts))
print("Val-Test overlap:", len(val_texts & test_texts))

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


In [291]:
X_train.head()

35531    us agency that could challenge trump jr stalle...
33292    trump fires back at obama says clinton unfit f...
27663    an angry morning joe skewers trump minion gene...
14731    trump committed to fair us elections free from...
36086    antizuma mp quits south africas corrupt anc jo...
Name: content, dtype: object

In [292]:
from tensorflow.keras.preprocessing.text import Tokenizer

VOCAB_SIZE = 20000

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

vocab_size = min(
    VOCAB_SIZE,
    len(tokenizer.word_index) + 1
)

print("Vocabulary Size:", vocab_size)

Vocabulary Size: 20000


In [293]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [294]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 300

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

In [295]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

In [296]:
class PositionalEncoding(layers.Layer):
    def __init__(self, max_len, d_model):
        super().__init__()

        pos = np.arange(max_len)[:, np.newaxis]
        i = np.arange(d_model)[np.newaxis, :]

        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
        angle_rads = pos * angle_rates

        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

        self.pos_encoding = tf.cast(angle_rads[np.newaxis, ...], tf.float32)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pos_encoding[:, :seq_len, :]

In [297]:
class TransformerEncoder(layers.Layer):
    def __init__(
        self,
        d_model,
        num_heads,
        ff_dim,
        rate=0.1
    ):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads
        )

        self.ffn = tf.keras.Sequential([
            layers.Dense(
                ff_dim,
                activation="relu"
            ),
            layers.Dense(d_model)
        ])

        self.layernorm1 = layers.LayerNormalization(
            epsilon=1e-6
        )

        self.layernorm2 = layers.LayerNormalization(
            epsilon=1e-6
        )

        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, x, training=False):

        # Multi-Head Self-Attention
        attention_output = self.attention(
            query=x,
            key=x,
            value=x,
            training=training
        )

        attention_output = self.dropout1(
            attention_output,
            training=training
        )

        # Residual Connection + Layer Normalization
        out1 = self.layernorm1(
            x + attention_output
        )

        # Feed Forward Network
        ffn_output = self.ffn(out1)

        ffn_output = self.dropout2(
            ffn_output,
            training=training
        )

        # Residual Connection + Layer Normalization
        return self.layernorm2(
            out1 + ffn_output
        )

In [298]:
class TransformerModel(tf.keras.Model):

    def __init__(
        self,
        vocab_size,
        max_len,
        d_model,
        num_heads,
        ff_dim,
        num_transformer_blocks,
        output_dim,
        rate=0.1
    ):
        super().__init__()

        self.embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=d_model,
        )

        self.pos_encoding = PositionalEncoding(
            max_len,
            d_model
        )

        self.encoder_blocks = [
            TransformerEncoder(
                d_model=d_model,
                num_heads=num_heads,
                ff_dim=ff_dim,
                rate=rate
            )
            for _ in range(num_transformer_blocks)
        ]

        self.dropout = layers.Dropout(rate)

        self.pooling = layers.GlobalAveragePooling1D()

        self.classifier = layers.Dense(
            output_dim,
            activation="softmax"
        )

    def call(self, inputs, training=False):

        # Embedding
        x = self.embedding(inputs)

        # Positional Encoding
        x = self.pos_encoding(x)

        x = self.dropout(
            x,
            training=training
        )

        # Transformer Encoder Blocks
        for encoder in self.encoder_blocks:

            x = encoder(
                x,
                training=training
            )

        # Global Average Pooling
        x = self.pooling(x)

        # Classification
        return self.classifier(x)

In [299]:
model = TransformerModel(
    vocab_size=vocab_size,
    max_len=MAX_LEN,
    d_model=128,
    num_heads=8,
    ff_dim=512,
    num_transformer_blocks=2,
    output_dim=2
)

In [300]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [301]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [302]:
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stopping]
)

Epoch 1/10
977/977 ━━━━━━━━━━━━━━━━━━━━ 361s 364ms/step - accuracy: 0.8999 - loss: 0.1894 - val_accuracy: 0.9992 - val_loss: 0.0035
Epoch 2/10
977/977 ━━━━━━━━━━━━━━━━━━━━ 415s 425ms/step - accuracy: 0.9972 - loss: 0.0099 - val_accuracy: 0.9987 - val_loss: 0.0060
Epoch 3/10
977/977 ━━━━━━━━━━━━━━━━━━━━ 359s 367ms/step - accuracy: 0.9980 - loss: 0.0074 - val_accuracy: 0.9987 - val_loss: 0.0076


In [303]:
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

123/123 ━━━━━━━━━━━━━━━━━━━━ 14s 115ms/step - accuracy: 0.9982 - loss: 0.0095
Test Loss: 0.00946800597012043
Test Accuracy: 0.9982088208198547


In [304]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_prob = model.predict(
    X_test_pad
)

y_pred = np.argmax(
    y_pred_prob,
    axis=1
)

123/123 ━━━━━━━━━━━━━━━━━━━━ 15s 117ms/step


In [305]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[1789    2]
 [   5 2112]]


In [306]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Fake", "True"]
    )
)

              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00      1791
        True       1.00      1.00      1.00      2117

    accuracy                           1.00      3908
   macro avg       1.00      1.00      1.00      3908
weighted avg       1.00      1.00      1.00      3908

